# Introduction to Linear Regression
## From Simple to Multiple Regression using Scikit-Learn

In this notebook we will:
1. Load and explore a used-car listing dataset
2. Build a **simple linear regression** (one predictor → price)
3. Extend to **multiple linear regression** (several predictors → price)
4. Evaluate every model with standard **error / prediction metrics**

We follow a machine-learning pipeline: **train / test split → fit → predict → evaluate**.

---
## 1 — Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load the dataset
df = pd.read_csv('../../../../OneDrive - University of Virginia/DS-3021/data/USA_cars_datasets.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

---
## 2 — Data Cleaning

We need numeric features to feed into a regression model.  
Key decisions:
- **Target variable (`y`)**: `price`
- **Numeric features available**: `year`, `mileage`
- Drop rows where `price` is 0 (likely data-entry errors)
- Remove extreme outlier prices to keep the model focused on typical listings

In [ ]:
# Drop rows with price = 0
df = df[df['price'] > 0].copy()

# Remove extreme price outliers (keep below 99th percentile)
upper = df['price'].quantile(0.99)
df = df[df['price'] <= upper].copy()

print(f"Cleaned dataset shape: {df.shape}")
df[['price', 'year', 'mileage']].describe()

### Quick Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['price'], bins=40, edgecolor='black')
axes[0].set_title('Distribution of Price')
axes[0].set_xlabel('Price ($)')

axes[1].scatter(df['mileage'], df['price'], alpha=0.3, s=10)
axes[1].set_title('Price vs Mileage')
axes[1].set_xlabel('Mileage')
axes[1].set_ylabel('Price ($)')

axes[2].scatter(df['year'], df['price'], alpha=0.3, s=10)
axes[2].set_title('Price vs Year')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()

---
## 3 — Simple Linear Regression

**Goal**: Predict `price` from a single feature — `year`.

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{year}$$

### ML Pipeline Steps
1. Select feature (`X`) and target (`y`)
2. Split into **training** and **testing** sets
3. Fit the model on training data
4. Predict on test data
5. Evaluate with error metrics

### 3.1 — Train / Test Split

In [ ]:
# Feature and target
X_simple = df[['year']]       # 2-D array (sklearn requirement)
y = df['price']

# 80/20 split, reproducible with random_state
X_train_s, X_test_s, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_s.shape[0]}")
print(f"Testing  set size: {X_test_s.shape[0]}")

### 3.2 — Fit the Model

In [ ]:
slr = LinearRegression()
slr.fit(X_train_s, y_train)

print(f"Intercept (β₀): {slr.intercept_:,.2f}")
print(f"Coefficient (β₁ for year): {slr.coef_[0]:,.2f}")
print(f"\nInterpretation: Each additional year is associated with a "
      f"${slr.coef_[0]:,.0f} change in predicted price.")

### 3.3 — Predict on the Test Set

In [ ]:
y_pred_s = slr.predict(X_test_s)

# Show a few predictions vs actuals
comparison = pd.DataFrame({
    'Year': X_test_s['year'].values,
    'Actual Price': y_test.values,
    'Predicted Price': np.round(y_pred_s, 2)
})
comparison.head(10)

### 3.4 — Error Metrics

| Metric | What it measures |
|--------|------------------|
| **MAE** (Mean Absolute Error) | Average absolute difference between predicted and actual values — easy to interpret in dollars |
| **MSE** (Mean Squared Error) | Average of squared errors — penalizes large errors more heavily |
| **RMSE** (Root MSE) | Square root of MSE — back in the same units as price |
| **R²** (Coefficient of Determination) | Proportion of variance in the target explained by the model (0 to 1) |

In [ ]:
def print_metrics(y_true, y_pred, model_name='Model'):
    """Calculate and display regression error metrics."""
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    
    print(f"=== {model_name} ===")
    print(f"  MAE  : ${mae:,.2f}")
    print(f"  MSE  : {mse:,.2f}")
    print(f"  RMSE : ${rmse:,.2f}")
    print(f"  R²   : {r2:.4f}")
    print()
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

simple_metrics = print_metrics(y_test, y_pred_s, 'Simple Linear Regression (year → price)')

### 3.5 — Visualize the Fit

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression line over scatter
axes[0].scatter(X_test_s, y_test, alpha=0.3, s=10, label='Actual')
# Plot the line using sorted values for a clean line
x_line = np.linspace(X_test_s['year'].min(), X_test_s['year'].max(), 100).reshape(-1, 1)
axes[0].plot(x_line, slr.predict(x_line), color='red', linewidth=2, label='Predicted')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Simple Linear Regression: Year → Price')
axes[0].legend()

# Residual plot
residuals_s = y_test - y_pred_s
axes[1].scatter(y_pred_s, residuals_s, alpha=0.3, s=10)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot (Simple LR)')

plt.tight_layout()
plt.show()

---
## 4 — Multiple Linear Regression

Can we improve predictions by using **more than one feature**?

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{year} + \beta_2 \cdot \text{mileage}$$

We follow the same pipeline: split → fit → predict → evaluate.

### 4.1 — Feature Selection & Correlation

In [ ]:
# Correlation matrix for the numeric columns we care about
corr_cols = ['price', 'year', 'mileage']
corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()

### 4.2 — Train / Test Split (Multiple Features)

In [ ]:
X_multi = df[['year', 'mileage']]

# Use the SAME random_state so train/test rows are identical
X_train_m, X_test_m, y_train, y_test = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_m.shape[0]}")
print(f"Testing  set size: {X_test_m.shape[0]}")

### 4.3 — Fit & Interpret

In [ ]:
mlr = LinearRegression()
mlr.fit(X_train_m, y_train)

print(f"Intercept (β₀): {mlr.intercept_:,.2f}")
for name, coef in zip(X_multi.columns, mlr.coef_):
    print(f"Coefficient for {name}: {coef:,.4f}")

print(f"\nInterpretation (holding other features constant):")
print(f"  • Each additional year  → ${mlr.coef_[0]:,.0f} change in price")
print(f"  • Each additional mile  → ${mlr.coef_[1]:.4f} change in price")

### 4.4 — Predict & Evaluate

In [ ]:
y_pred_m = mlr.predict(X_test_m)

multi_metrics = print_metrics(y_test, y_pred_m, 'Multiple Linear Regression (year + mileage → price)')

### 4.5 — Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_m, alpha=0.3, s=10)
max_val = max(y_test.max(), y_pred_m.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Predicted vs Actual (Multiple LR)')
axes[0].legend()

# Residuals
residuals_m = y_test - y_pred_m
axes[1].scatter(y_pred_m, residuals_m, alpha=0.3, s=10)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot (Multiple LR)')

plt.tight_layout()
plt.show()

---
## 5 — Adding Categorical Features (Brand)

Linear regression can also handle categorical variables via **one-hot encoding**.  
Let's add the car `brand` as a third predictor to see if it improves the model.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Keep only brands that appear frequently enough to be meaningful
top_brands = df['brand'].value_counts().head(10).index.tolist()
df_top = df[df['brand'].isin(top_brands)].copy()
print(f"Rows after filtering to top 10 brands: {len(df_top)}")
print(f"Brands: {top_brands}")

In [ ]:
X_full = df_top[['year', 'mileage', 'brand']]
y_full = df_top['price']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_f.shape[0]}")
print(f"Testing  set size: {X_test_f.shape[0]}")

### 5.1 — Scikit-Learn Pipeline

A `Pipeline` bundles preprocessing and modeling into a single object.  
This prevents **data leakage** — the encoder only learns categories from the training set.

In [ ]:
# Define which columns get which treatment
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', ['year', 'mileage']),          # numeric → no change
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), ['brand'])  # categorical → one-hot
    ]
)

# Full pipeline: preprocess → linear regression
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

pipe.fit(X_train_f, y_train_f)
print("Pipeline fitted successfully.")

In [ ]:
y_pred_f = pipe.predict(X_test_f)

full_metrics = print_metrics(y_test_f, y_pred_f, 'Multiple LR with Brand (Pipeline)')

In [ ]:
# Predicted vs Actual for the full model
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_f, y_pred_f, alpha=0.3, s=10)
max_val = max(y_test_f.max(), y_pred_f.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Predicted vs Actual (Year + Mileage + Brand)')
axes[0].legend()

residuals_f = y_test_f - y_pred_f
axes[1].hist(residuals_f, bins=40, edgecolor='black')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Residuals (Full Model)')

plt.tight_layout()
plt.show()

---
## 6 — Model Comparison

Let's compare all three models side-by-side.

In [ ]:
results = pd.DataFrame({
    'Simple LR (year)': simple_metrics,
    'Multiple LR (year + mileage)': multi_metrics,
    'Multiple LR + Brand (pipeline)': full_metrics
}).T

results.style.format({
    'MAE': '${:,.2f}',
    'MSE': '{:,.0f}',
    'RMSE': '${:,.2f}',
    'R2': '{:.4f}'
})

In [ ]:
# Visual comparison of R² and RMSE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = ['Simple LR\n(year)', 'Multiple LR\n(year + mileage)', 'Multiple LR\n+ Brand']
r2_values  = [simple_metrics['R2'], multi_metrics['R2'], full_metrics['R2']]
rmse_values = [simple_metrics['RMSE'], multi_metrics['RMSE'], full_metrics['RMSE']]

colors = ['#3498db', '#2ecc71', '#e74c3c']

axes[0].bar(model_names, r2_values, color=colors, edgecolor='black')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Comparison (higher is better)')
axes[0].set_ylim(0, 1)
for i, v in enumerate(r2_values):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(model_names, rmse_values, color=colors, edgecolor='black')
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('RMSE Comparison (lower is better)')
for i, v in enumerate(rmse_values):
    axes[1].text(i, v + 200, f'${v:,.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 7 — Key Takeaways

### What We Learned

1. **Simple Linear Regression** uses one feature to predict the target. It is easy to visualize and interpret but may miss important information.

2. **Multiple Linear Regression** uses two or more features. Adding `mileage` alongside `year` should improve predictions because both carry information about a car's value.

3. **Categorical features** (like `brand`) can be incorporated via **one-hot encoding**. A scikit-learn `Pipeline` bundles encoding and modeling so the encoder only sees training data, preventing data leakage.

4. **Error metrics** tell us how well the model predicts:
   - **MAE / RMSE** — interpretable in dollars; lower is better
   - **R²** — proportion of variance explained; higher (closer to 1) is better

5. **Train / test splitting** is essential: we evaluate on data the model has *never seen* to estimate real-world performance.

### The ML Pipeline Pattern

```
Raw Data → Clean / Engineer Features → Train/Test Split → Fit Model → Predict → Evaluate
```

This pattern generalizes to virtually every supervised learning problem you will encounter.